<a href="https://colab.research.google.com/github/RobotMa/UniAD/blob/v2.0-qianli/UniAD_Eval_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UniAD 2.0 Evaluation on Google Colab

This notebook runs UniAD Stage 1 (Track & Map) evaluation on Colab's free T4 GPU.

**Setup:**
- Uses a pre-built conda environment (Python 3.9 + PyTorch 2.0.1 + mmcv/mmdet/mmseg/mmdet3d)
- Packaged via `conda-pack` and stored on Google Drive
- No building from source on Colab — everything is pre-built

**Prerequisites:**
1. Build the conda environment locally following [docs/INSTALL.md](docs/INSTALL.md)
2. Package it: `conda pack -n uniad2.0 -o uniad_env.tar.gz`
3. Upload `uniad_env.tar.gz` to `Google Drive/colab_cache/UniAD/`
4. Upload nuScenes dataset to `Google Drive/colab_cache/UniAD/nuscenes/`

**Estimated Time:**
- Setup: ~2 min (extract cached env)
- Evaluation: ~2-4 hours on T4

## 1. Check GPU

In [ ]:
# Verify GPU is available
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
import os
from google.colab import drive

# Mount Google Drive (handles already-mounted case gracefully)
if os.path.ismount('/content/drive'):
    print("Google Drive already mounted at /content/drive")
else:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")

# Create cache directories on Google Drive for persistence
CACHE_DIR = "/content/drive/MyDrive/colab_cache/UniAD"
!mkdir -p {CACHE_DIR}
!mkdir -p /content/drive/MyDrive/colab_cache/UniAD_ckpts
print("Cache directories ready.")

## 3. Clone UniAD Repository

In [ ]:
import os
import subprocess

%cd /content

# Clone UniAD repo only if it doesn't exist
if os.path.exists('/content/UniAD'):
    print("UniAD repository already exists, updating...")
    %cd UniAD
    
    # Check for local changes before resetting
    result = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True)
    if result.stdout.strip():
        print("\n⚠️  WARNING: You have local changes that will be preserved:")
        print(result.stdout)
        print("Stashing local changes...")
        !git stash
        stashed = True
    else:
        stashed = False
    
    !git fetch origin
    !git checkout v2.0-qianli
    !git reset --hard origin/v2.0-qianli
    
    # Restore stashed changes if any
    if stashed:
        print("\nRestoring your local changes...")
        !git stash pop
        print("✓ Local changes restored")
else:
    print("Cloning UniAD repository...")
    !git clone -b v2.0-qianli https://github.com/RobotMa/UniAD.git
    %cd UniAD

print(f"\nWorking directory: {os.getcwd()}")
!git branch --show-current

## 4. Restore Conda Environment

This extracts the pre-built conda environment from Google Drive (~2 min).

**If you haven't created it yet**, follow [docs/INSTALL.md](docs/INSTALL.md):
```bash
# On your local machine:
conda create -n uniad2.0 python=3.9 -y && conda activate uniad2.0
pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
pip install mmcv-full==1.6.1 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
pip install mmdet==2.26.0 mmsegmentation==0.29.1 mmdet3d==1.0.0rc6
pip install nuscenes-devkit motmetrics einops casadi pytorch-lightning torchmetrics
conda install -c conda-forge conda-pack && conda pack -n uniad2.0 -o uniad_env.tar.gz
```
Then upload `uniad_env.tar.gz` to `Google Drive/colab_cache/UniAD/`.

In [ ]:
import os
import subprocess

CACHE_DIR = "/content/drive/MyDrive/colab_cache/UniAD"
ENV_TARBALL = f"{CACHE_DIR}/uniad_env.tar.gz"
ENV_PATH = "/content/uniad_env"
PYTHON = f"{ENV_PATH}/bin/python"

# Check that the conda-pack tarball exists on Drive
if not os.path.exists(ENV_TARBALL):
    raise FileNotFoundError(
        f"Conda environment not found at: {ENV_TARBALL}\n"
        "Please build it locally and upload to Google Drive.\n"
        "See the instructions in the cell above."
    )

# Extract if not already done
if os.path.exists(f"{ENV_PATH}/bin/python"):
    print("✓ Conda environment already extracted")
else:
    print(f"Extracting conda environment (~2 min)...")
    os.makedirs(ENV_PATH, exist_ok=True)
    result = subprocess.run(
        f"tar -xzf {ENV_TARBALL} -C {ENV_PATH}",
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to extract environment:\n{result.stderr}")
    
    # Fix conda-pack prefixes
    print("Fixing environment prefixes...")
    subprocess.run(
        f"source {ENV_PATH}/bin/activate && conda-unpack",
        shell=True, executable="/bin/bash"
    )
    print("✓ Conda environment extracted and ready")

# Fix Python 3.9 compatibility: fractions.gcd was removed in 3.9 but old
# networkx (<2.6) still imports it. Patch via sitecustomize.py so it runs
# before any imports.
site_pkg = f"{ENV_PATH}/lib/python3.9/site-packages"
sitecustomize = f"{site_pkg}/sitecustomize.py"
patch_code = "import fractions, math\nif not hasattr(fractions, 'gcd'): fractions.gcd = math.gcd\n"

if not os.path.exists(sitecustomize) or patch_code not in open(sitecustomize).read():
    print("Patching fractions.gcd for Python 3.9 compatibility...")
    with open(sitecustomize, "a") as f:
        f.write(patch_code)
    print("✓ sitecustomize.py patched")
else:
    print("✓ fractions.gcd patch already applied")

# Verify key packages
print(f"\nEnvironment Python: {PYTHON}")
result = subprocess.run(
    [PYTHON, "-c",
     "import torch, mmcv, mmdet, mmseg, mmdet3d; "
     "print(f'torch: {torch.__version__}'); "
     "print(f'mmcv: {mmcv.__version__}'); "
     "print(f'mmdet: {mmdet.__version__}'); "
     "print(f'mmseg: {mmseg.__version__}'); "
     "print(f'mmdet3d: {mmdet3d.__version__}'); "
     "print(f'CUDA available: {torch.cuda.is_available()}')"
    ],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(result.stdout)
    print("✓ All packages verified!")
else:
    print(f"✗ Verification failed:\n{result.stderr}")
    raise RuntimeError("Conda environment is broken. Please rebuild and re-upload.")

## 5. Download Pretrained Checkpoints

In [ ]:
import os

%cd /content/UniAD
!mkdir -p ckpts

CKPT_CACHE = "/content/drive/MyDrive/colab_cache/UniAD_ckpts"

# Ensure checkpoint cache directory exists
!mkdir -p {CKPT_CACHE}

# Check if checkpoints already cached on Drive
if os.path.exists(f"{CKPT_CACHE}/uniad_base_track_map.pth"):
    print("Found cached checkpoints on Drive! Linking...")
    !ln -sf {CKPT_CACHE}/uniad_base_track_map.pth ckpts/
    !ln -sf {CKPT_CACHE}/bevformer_r101_dcn_24ep.pth ckpts/
else:
    print("Downloading checkpoints (will be cached to Drive)...")
    %cd {CKPT_CACHE}
    !wget -q --show-progress https://huggingface.co/OpenDriveLab/UniAD2.0_R101_nuScenes/resolve/main/ckpts/uniad_base_track_map.pth
    !wget -q --show-progress https://huggingface.co/OpenDriveLab/UniAD2.0_R101_nuScenes/resolve/main/ckpts/bevformer_r101_dcn_24ep.pth
    %cd /content/UniAD
    !ln -sf {CKPT_CACHE}/uniad_base_track_map.pth ckpts/
    !ln -sf {CKPT_CACHE}/bevformer_r101_dcn_24ep.pth ckpts/

print("\nCheckpoints ready:")
!ls -lh ckpts/

## 6. Setup nuScenes Dataset (Cached on Drive)

The nuScenes dataset is large (~4 GB for mini, ~300+ GB for full trainval). Caching it on Google Drive avoids re-downloading every session.

### Step-by-step: Cache nuScenes on Google Drive

1. **Register** at https://www.nuscenes.org/sign-up (free academic account)

2. **Download** from https://www.nuscenes.org/download:

   **For mini (testing only):**
   | File | Size | Notes |
   |------|------|-------|
   | `v1.0-mini.tgz` | ~4 GB | All sensor data + metadata in one archive |
   | `can_bus.zip` | ~300 MB | Required |
   | `nuScenes-map-expansion-v1.3.zip` | ~500 MB | Required |

   **For full trainval (reproducing paper results):**
   | File | Size | Notes |
   |------|------|-------|
   | `v1.0-trainval_meta.tgz` | ~600 MB | Metadata (annotations, calibrations) |
   | `v1.0-trainval0X_keyframes.tgz` (1-10) | ~2-3 GB each | Camera images (`samples/CAM_*`) |
   | `v1.0-trainval0X_blobs_lidar.tgz` (1-10) | ~3 GB each | LiDAR point clouds (required for PKL generation) |
   | `can_bus.zip` | ~300 MB | Required |
   | `nuScenes-map-expansion-v1.3.zip` | ~500 MB | Required |

   > **Note:** Radar blobs are NOT required for UniAD (camera-only model).

3. **Extract** into one directory:
   ```bash
   mkdir -p nuscenes && cd nuscenes
   # Mini:
   tar -xzf v1.0-mini.tgz
   # Or full trainval:
   tar -xzf v1.0-trainval_meta.tgz
   for f in v1.0-trainval*_keyframes.tgz; do tar -xzf "$f"; done
   for f in v1.0-trainval*_blobs_lidar.tgz; do tar -xzf "$f"; done
   # Required extras:
   unzip can_bus.zip
   unzip nuScenes-map-expansion-v1.3.zip -d maps/
   ```

4. **Upload** to Google Drive:
   ```bash
   rclone copy ./nuscenes gdrive:colab_cache/UniAD/nuscenes --progress
   ```

**Expected structure on Drive:**
```
My Drive/colab_cache/UniAD/nuscenes/
├── can_bus/              # CAN bus expansion (required)
├── maps/                 # Map expansion (required)
├── samples/
│   ├── CAM_FRONT/        # Camera keyframes
│   ├── LIDAR_TOP/        # LiDAR keyframes (required for PKL generation)
│   └── ...
├── sweeps/
│   ├── LIDAR_TOP/        # LiDAR sweeps
│   └── ...
├── v1.0-mini/            # (if using mini)
└── v1.0-trainval/        # (if using full dataset)
```

**Notes:**
- **Mini** (~4 GB): Good for testing the pipeline. Results will NOT match paper metrics.
- **Full trainval**: Required for reproducing paper results (AMOTA 0.394).
- **CAN bus** and **LiDAR** data are required for all dataset versions (PKL generation needs them).
- Once uploaded, the dataset persists across Colab sessions.

In [ ]:
import os

CACHE_DIR = "/content/drive/MyDrive/colab_cache/UniAD"
NUSCENES_CACHE = f"{CACHE_DIR}/nuscenes"

# Create data directory
!mkdir -p /content/UniAD/data

# Check if nuScenes exists on Drive
if os.path.exists(NUSCENES_CACHE) and os.listdir(NUSCENES_CACHE):
    print("✓ Found nuScenes on Google Drive!")
    !ln -sf {NUSCENES_CACHE} /content/UniAD/data/nuscenes
    print("  Linked to /content/UniAD/data/nuscenes")
    
    # Check contents
    print("\n  Contents:")
    !ls /content/UniAD/data/nuscenes/ | head -10
    
    # Warn if using mini dataset
    if os.path.exists(f"{NUSCENES_CACHE}/v1.0-mini") and not os.path.exists(f"{NUSCENES_CACHE}/v1.0-trainval"):
        print("\n⚠️  WARNING: Using nuScenes MINI dataset.")
        print("   Evaluation results will NOT match paper metrics.")
        print("   For valid results, upload the full trainval dataset.")
else:
    print("="*60)
    print("nuScenes dataset not found on Google Drive!")
    print("="*60)
    print("\nPlease download nuScenes manually:")
    print("1. Go to: https://www.nuscenes.org/download")
    print("2. Register for a free account")
    print("3. Download 'Mini' (for testing) or 'Trainval' (for full eval)")
    print("4. Upload to Google Drive at:")
    print(f"   {NUSCENES_CACHE}/")
    print("\nExpected structure after upload:")
    print("   nuscenes/")
    print("   ├── maps/")
    print("   ├── samples/")
    print("   ├── sweeps/")
    print("   └── v1.0-mini/ (or v1.0-trainval)")
    print("\nThen re-run this cell.")
    print("="*60)
    
    # Create the directory so user knows where to upload
    !mkdir -p {NUSCENES_CACHE}
    print(f"\nCreated empty directory: {NUSCENES_CACHE}")
    print("Upload your nuScenes data there.")

## 7. Prepare Data PKL Files (Auto-Cached)

PKL files contain preprocessed metadata (annotations, calibrations). They're generated once and cached to Drive.

In [ ]:
import os
import glob

%cd /content/UniAD

ENV_PATH = "/content/uniad_env"
PYTHON = f"{ENV_PATH}/bin/python"

# Check if PKL files already exist (cached from previous run)
pkl_files = glob.glob("/content/UniAD/data/nuscenes/*_infos_*.pkl")

if pkl_files:
    print("✓ PKL files already exist (cached):")
    for f in pkl_files:
        print(f"  {os.path.basename(f)}")
else:
    # Verify CAN bus data exists
    if not os.path.exists("/content/UniAD/data/nuscenes/can_bus"):
        raise FileNotFoundError(
            "CAN bus data not found at data/nuscenes/can_bus/\n"
            "Please download can_bus.zip from https://www.nuscenes.org/download\n"
            "and extract it into your nuscenes directory on Google Drive."
        )
    
    print("✗ PKL files not found. Generating (one-time, ~10-20 min)...")
    print("  These will be cached to Drive for next time.\n")
    
    # Detect dataset version
    NUSCENES_PATH = "/content/UniAD/data/nuscenes"
    if os.path.exists(f"{NUSCENES_PATH}/v1.0-mini") and not os.path.exists(f"{NUSCENES_PATH}/v1.0-trainval"):
        version = "v1.0-mini"
        print(f"  Detected dataset version: {version}")
    else:
        version = "v1.0"
        print(f"  Detected dataset version: {version} (trainval + test)")
    
    !{PYTHON} tools/create_data.py nuscenes \
        --root-path ./data/nuscenes \
        --canbus ./data/nuscenes \
        --out-dir ./data/nuscenes \
        --extra-tag nuscenes \
        --version {version}
    
    print("\n✓ PKL files generated and cached!")
    
    # Show generated files
    pkl_files = glob.glob("/content/UniAD/data/nuscenes/*_infos_*.pkl")
    for f in pkl_files:
        print(f"  {os.path.basename(f)}")

## 8. Run Evaluation

In [ ]:
import os

%cd /content/UniAD

ENV_PATH = "/content/uniad_env"
PYTHON = f"{ENV_PATH}/bin/python"

# Detect dataset type and warn if using mini
NUSCENES_PATH = "/content/UniAD/data/nuscenes"
using_mini = os.path.exists(f"{NUSCENES_PATH}/v1.0-mini") and not os.path.exists(f"{NUSCENES_PATH}/v1.0-trainval")

if using_mini:
    print("="*60)
    print("⚠️  WARNING: Running evaluation on nuScenes MINI dataset")
    print("="*60)
    print("Results will NOT match paper metrics (AMOTA 0.394).")
    print("Mini dataset is for testing the pipeline only.")
    print("For valid evaluation, use the full trainval dataset.")
    print("="*60 + "\n")

# Run Stage 1 evaluation with 1 GPU (using conda env's Python)
!{PYTHON} tools/test.py \
    projects/configs/stage1_track_map/base_track_map.py \
    ckpts/uniad_base_track_map.pth \
    --eval bbox

## Expected Results

If everything works correctly, you should see:
```
Aggregated results:
AMOTA    0.394
AMOTP    1.316
RECALL   0.484
```

## Troubleshooting

**"Conda environment not found" error:**
- Build it locally: see [docs/INSTALL.md](docs/INSTALL.md)
- Package it: `conda pack -n uniad2.0 -o uniad_env.tar.gz`
- Upload to: `Google Drive/colab_cache/UniAD/uniad_env.tar.gz`

**"CUDA not available" after extracting environment:**
- Colab's CUDA driver must be compatible with CUDA 11.8 in the environment
- Ensure GPU runtime is enabled: Runtime -> Change runtime type -> T4 GPU

**Out of Memory:**
- Try reducing `queue_length` from 5 to 3 in the config
- Use Colab Pro for A100 GPU

**Missing files:**
- Ensure nuScenes dataset is properly linked
- Check that all .pkl files exist in data/nuscenes/

**Session disconnected:**
- Colab sessions timeout after idle periods
- The conda environment on Drive persists — just re-run from Section 2